In [1]:
import piplite
await piplite.install(['folium'])
await piplite.install(['pandas'])

In [2]:
import folium
import pandas as pd

In [3]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

In [4]:
# Download and read the `spacex_launch_geo.csv`
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df=pd.read_csv(spacex_csv_file)

In [5]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [6]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

In [7]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

In [10]:
# 1. Initialize the map centered around the NASA coordinate
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# 2. Loop through each launch site to add the Circle and Marker
for index, row in launch_sites_df.iterrows():
    # Extract the coordinate and label for the current launch site
    coordinate = [row['Lat'], row['Long']]
    site_name = row['Launch Site']
    
    # Create and add the folium.Circle (1km radius)
    circle = folium.Circle(
        coordinate, 
        radius=1000, 
        color='#d35400', 
        fill=True
    ).add_child(folium.Popup(site_name))
    
    # Create and add the folium.Marker text label
    marker = folium.map.Marker(
        coordinate, 
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0), 
            html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>' % site_name, 
        )
    )
    
    # Add both elements directly to your newly initialized site_map
    site_map.add_child(circle)
    site_map.add_child(marker)

# 3. Display the completed map
site_map

In [11]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


In [12]:
marker_cluster = MarkerCluster()


In [13]:
# Apply a function to check the value of `class` column
# If class=1, marker_color value will be green
# If class=0, marker_color value will be red
spacex_df['marker_color'] = spacex_df['class'].map({
    1: 'green',
    0: 'red'
})

In [14]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


In [15]:
# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

# for each row in spacex_df data frame
# create a Marker object with its coordinate
# and customize the Marker's icon property to indicate if this launch was successed or failed, 
# e.g., icon=folium.Icon(color='white', icon_color=row['marker_color']
for index, record in spacex_df.iterrows():
    # Create a Marker and add it to the MarkerCluster
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        icon=folium.Icon(
            icon='info-sign',
            color='white',
            icon_color=record['marker_color']
        ),
        popup = folium.Popup(
            f"Launch Site: {record['Launch Site']}<br>"
            f"Class: {'Success' if record['class'] == 1 else 'Failure'}",
            max_width=300
        )
    )
    marker_cluster.add_child(marker)

site_map

In [16]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

In [17]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

In [18]:
# find coordinate of the closet coastline
# e.g.,: Lat: 28.56367  Lon: -80.57163
# distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
# 
launch_site_lat = 28.563197
launch_site_lon = -80.576820
coastline_lat = 28.56289
coastline_lon = -80.56755
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
distance_coastline

0.9062504780818508

In [19]:
distance_marker = folium.Marker(
    location = [coastline_lat, coastline_lon],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
        )
    )

site_map.add_child(distance_marker)


In [20]:
# Create a `folium.PolyLine` object using the coastline coordinates and launch site coordinate
# lines=folium.PolyLine(locations=coordinates, weight=1)

lines=folium.PolyLine(locations = [[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]], weight=1)
site_map.add_child(lines)

In [21]:
def plot_distance_lines(launchsite_lat, launchsite_lon, proximity_lat, proximity_lon):
    # the distance from launch site to proximities
    distance_proximity = calculate_distance(launchsite_lat, launchsite_lon, proximity_lat, proximity_lon)
    print(distance_proximity)

    # plot the distance in map
    distance_marker = folium.Marker(
    location = [proximity_lat, proximity_lon],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_proximity),
        )
    )
    site_map.add_child(distance_marker)
    
    # draw a line from launch site to proximities
    lines = folium.PolyLine(locations = [[launchsite_lat, launchsite_lon], [proximity_lat, proximity_lon]], weight=1)
    site_map.add_child(lines)

    return site_map

In [22]:
launchsite_lat = 28.563197
launchsite_lon = -80.576820
railway_lat = 28.56364
railway_lon = -80.57473
highway_lat = 28.56357
highway_lon = -80.57084
city_lat = 28.60803
city_lon = -80.81543

In [23]:
plot_distance_lines(launchsite_lat, launchsite_lon, railway_lat, railway_lon)

0.21003774478930085


In [24]:
plot_distance_lines(launchsite_lat, launchsite_lon, highway_lat, highway_lon)

0.5856689841983389


In [25]:
plot_distance_lines(launchsite_lat, launchsite_lon, city_lat, city_lon)

23.832887470082607
